# 04 Report Exports

Export CSV tables and optional SVG figures from the standardized Task 1 outputs.


In [ ]:
from pathlib import Path
import csv
import json
import math
import os
import sys
from xml.sax.saxutils import escape

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

ROOT


In [ ]:
SCENARIO_ORDER = {"real": 0, "real_aug": 1, "synth": 2, "both": 3}
DEFAULT_SPLITS = ("train", "val", "test")


def iter_files_count(dir_path):
    dir_path = Path(dir_path)
    if not dir_path.exists():
        return 0
    return sum(1 for path in dir_path.rglob("*") if path.is_file())


def export_dataset_summary(data_root, out_csv, splits=DEFAULT_SPLITS):
    data_root = Path(data_root)
    out_csv = Path(out_csv)
    rows = []
    for split in splits:
        split_dir = data_root / split
        if not split_dir.exists():
            continue
        class_dirs = sorted([path for path in split_dir.iterdir() if path.is_dir()], key=lambda path: path.name)
        for class_dir in class_dirs:
            rows.append({"split": split, "class": class_dir.name, "n_images": iter_files_count(class_dir)})
    out_csv.parent.mkdir(parents=True, exist_ok=True)
    with out_csv.open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["split", "class", "n_images"])
        writer.writeheader()
        writer.writerows(rows)
    return rows


def export_gan_fid(train_log_json, out_csv):
    train_log_json = Path(train_log_json)
    out_csv = Path(out_csv)
    rows = []
    for item in json.loads(train_log_json.read_text()):
        if "fid" in item:
            rows.append({"epoch": int(item["epoch"]), "fid": float(item["fid"])})
    rows.sort(key=lambda row: row["epoch"])
    out_csv.parent.mkdir(parents=True, exist_ok=True)
    with out_csv.open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["epoch", "fid"])
        writer.writeheader()
        writer.writerows(rows)
    return rows


def _n_per_class_key(value):
    if isinstance(value, int):
        return (0, value)
    try:
        return (0, int(value))
    except Exception:
        return (1, str(value))


def export_clf_results(all_results_json, out_csv):
    all_results_json = Path(all_results_json)
    out_csv = Path(out_csv)
    rows = []
    for result in json.loads(all_results_json.read_text()):
        per_class = result.get("per_class", {}) or {}
        rows.append(
            {
                "n_per_class": result.get("n_per_class"),
                "scenario": result.get("scenario"),
                "augmentation_policy": result.get("augmentation_policy"),
                "train_size": result.get("train_size"),
                "test_accuracy": result.get("test_accuracy"),
                "train_time_sec": result.get("train_time_sec"),
                "classifier_train_time_sec": result.get("classifier_train_time_sec", result.get("train_time_sec")),
                "gan_train_time_sec": result.get("gan_train_time_sec", 0.0),
                "synth_generation_time_sec": result.get("synth_generation_time_sec", 0.0),
                "pipeline_time_sec": result.get("pipeline_time_sec", result.get("train_time_sec")),
                "apple_f1": (per_class.get("apple", {}) or {}).get("f1"),
                "banana_f1": (per_class.get("banana", {}) or {}).get("f1"),
                "orange_f1": (per_class.get("orange", {}) or {}).get("f1"),
            }
        )
    rows.sort(key=lambda row: (_n_per_class_key(row["n_per_class"]), SCENARIO_ORDER.get(row["scenario"], 999)))
    out_csv.parent.mkdir(parents=True, exist_ok=True)
    with out_csv.open("w", newline="") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=[
                "n_per_class",
                "scenario",
                "augmentation_policy",
                "train_size",
                "test_accuracy",
                "train_time_sec",
                "classifier_train_time_sec",
                "gan_train_time_sec",
                "synth_generation_time_sec",
                "pipeline_time_sec",
                "apple_f1",
                "banana_f1",
                "orange_f1",
            ],
        )
        writer.writeheader()
        writer.writerows(rows)
    return rows


In [ ]:
def load_fid_points(train_log_json):
    train_log_json = Path(train_log_json)
    points = []
    for item in json.loads(train_log_json.read_text()):
        if "fid" in item:
            points.append((int(item["epoch"]), float(item["fid"])))
    points.sort(key=lambda pair: pair[0])
    return points


def nice_ticks(vmin, vmax, n=5):
    if not math.isfinite(vmin) or not math.isfinite(vmax) or vmin == vmax:
        return [vmin]
    span = vmax - vmin
    raw_step = span / max(1, n - 1)
    power = 10 ** math.floor(math.log10(raw_step))
    step = power
    for multiple in (1, 2, 5, 10):
        step = multiple * power
        if step >= raw_step:
            break
    start = math.floor(vmin / step) * step
    end = math.ceil(vmax / step) * step
    ticks = []
    value = start
    while value <= end + 1e-12:
        ticks.append(value)
        value += step
    return ticks


def render_svg(points, title):
    if not points:
        raise ValueError("No FID points found in train log.")
    width, height = 900, 520
    margin = {"l": 80, "r": 30, "t": 60, "b": 70}
    plot_w = width - margin["l"] - margin["r"]
    plot_h = height - margin["t"] - margin["b"]
    epochs = [point[0] for point in points]
    fids = [point[1] for point in points]
    x_min, x_max = min(epochs), max(epochs)
    y_min, y_max = min(fids), max(fids)
    y_pad = 0.05 * (y_max - y_min) if y_max > y_min else 1.0
    y0, y1 = y_min - y_pad, y_max + y_pad

    def x_to_px(x):
        if x_max == x_min:
            return margin["l"] + plot_w / 2
        return margin["l"] + (x - x_min) / (x_max - x_min) * plot_w

    def y_to_px(y):
        if y1 == y0:
            return margin["t"] + plot_h / 2
        return margin["t"] + (y1 - y) / (y1 - y0) * plot_h

    polyline = " ".join(f"{x_to_px(epoch):.2f},{y_to_px(fid):.2f}" for epoch, fid in points)
    best_epoch, best_fid = min(points, key=lambda pair: pair[1])
    best_x, best_y = x_to_px(best_epoch), y_to_px(best_fid)
    x_ticks = nice_ticks(x_min, x_max, n=6)
    y_ticks = nice_ticks(y0, y1, n=6)

    def fmt_tick(value):
        if abs(value) >= 100:
            return f"{value:.0f}"
        if abs(value) >= 10:
            return f"{value:.1f}"
        return f"{value:.2f}"

    parts = []
    parts.append(f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">')
    parts.append('<rect x="0" y="0" width="100%" height="100%" fill="white"/>')
    parts.append(f'<text x="{width / 2:.1f}" y="32" text-anchor="middle" font-family="Inter, Arial, sans-serif" font-size="18" fill="#111">{escape(title)}</text>')
    parts.append(f'<text x="{width / 2:.1f}" y="52" text-anchor="middle" font-family="Inter, Arial, sans-serif" font-size="12" fill="#555">Lower is better</text>')
    for tick in y_ticks:
        y = y_to_px(tick)
        parts.append(f'<line x1="{margin["l"]}" y1="{y:.2f}" x2="{width - margin["r"]}" y2="{y:.2f}" stroke="#eee" stroke-width="1"/>')
        parts.append(f'<text x="{margin["l"] - 10}" y="{y + 4:.2f}" text-anchor="end" font-family="Inter, Arial, sans-serif" font-size="11" fill="#444">{escape(fmt_tick(tick))}</text>')
    for tick in x_ticks:
        x = x_to_px(tick)
        parts.append(f'<line x1="{x:.2f}" y1="{margin["t"]}" x2="{x:.2f}" y2="{height - margin["b"]}" stroke="#f3f3f3" stroke-width="1"/>')
        parts.append(f'<text x="{x:.2f}" y="{height - margin["b"] + 22}" text-anchor="middle" font-family="Inter, Arial, sans-serif" font-size="11" fill="#444">{escape(fmt_tick(tick))}</text>')
    parts.append(f'<rect x="{margin["l"]}" y="{margin["t"]}" width="{plot_w}" height="{plot_h}" fill="none" stroke="#444" stroke-width="1.2"/>')
    parts.append(f'<polyline fill="none" stroke="#1976D2" stroke-width="2.5" points="{polyline}"/>')
    parts.append(f'<circle cx="{best_x:.2f}" cy="{best_y:.2f}" r="5" fill="#D32F2F"/>')
    parts.append(f'<text x="{best_x + 10:.2f}" y="{best_y - 10:.2f}" font-family="Inter, Arial, sans-serif" font-size="12" fill="#D32F2F">best: epoch {best_epoch}, FID {best_fid:.2f}</text>')
    parts.append(f'<text x="{width / 2:.1f}" y="{height - 18}" text-anchor="middle" font-family="Inter, Arial, sans-serif" font-size="12" fill="#333">Epoch</text>')
    parts.append(f'<text x="18" y="{height / 2:.1f}" transform="rotate(-90 18 {height / 2:.1f})" text-anchor="middle" font-family="Inter, Arial, sans-serif" font-size="12" fill="#333">FID</text>')
    parts.append('</svg>')
    return "\n".join(parts)


def write_fid_svg(train_log_json, out_path, title="FID vs Epoch (CGAN Training)"):
    train_log_json = Path(train_log_json)
    out_path = Path(out_path)
    svg = render_svg(load_fid_points(train_log_json), title)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(svg)
    return out_path


def resolve_default_gan_log(task1_root):
    task1_root = Path(task1_root)
    candidates = sorted(
        task1_root.glob("gan/n*/train_log.json"),
        key=lambda path: int(path.parent.name[1:]) if path.parent.name.startswith("n") else -1,
    )
    if candidates:
        return candidates[-1]
    fallback = ROOT / "runs" / "gan" / "train_log.json"
    return fallback


In [ ]:
REPORTS_DIR = ROOT / "reports"
TABLES_DIR = REPORTS_DIR / "tables"
FIGURES_DIR = REPORTS_DIR / "figures"
TASK1_ROOT = ROOT / "runs" / "task1"
GAN_LOG = resolve_default_gan_log(TASK1_ROOT)
CLF_RESULTS = TASK1_ROOT / "clf" / "all_results.json"
if not CLF_RESULTS.exists():
    CLF_RESULTS = ROOT / "runs" / "clf" / "all_results.json"
DATASET_ROOT = ROOT / "data_final"

GAN_LOG, CLF_RESULTS


In [ ]:
# Uncomment the actions you want to run.
# dataset_rows = export_dataset_summary(DATASET_ROOT, TABLES_DIR / "dataset_summary.csv")
# clf_rows = export_clf_results(CLF_RESULTS, TABLES_DIR / "clf_results.csv")
# fid_rows = export_gan_fid(GAN_LOG, TABLES_DIR / "gan_fid_by_epoch.csv") if GAN_LOG.exists() else []
# if GAN_LOG.exists() and load_fid_points(GAN_LOG):
#     write_fid_svg(GAN_LOG, FIGURES_DIR / "fid_vs_epoch.svg")
